# Transformer Summarization from Scratch

This notebook implements a complete encoder-decoder Transformer summarizer in PyTorch for CNN/DailyMail using a custom BPE tokenizer, dynamic padding, masking, teacher forcing, label smoothing, greedy decoding, ROUGE evaluation, and error analysis. Seed: **42**.

## Section 1: Dataset Loading
Downloads CNN/DailyMail from Hugging Face and displays an article and reference summary. The subset sizes are intentionally small so `Run All` is reproducible on CPU while preserving all required training/evaluation steps.

In [ ]:
# If running in a fresh environment, uncomment the next line.
# %pip install -q -r requirements.txt

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'src'))

import torch
from summarization_project.config import ExperimentConfig, SpecialTokens
from summarization_project.data import (
    SummarizationDataset,
    load_cnn_dailymail_splits,
    make_dataloader,
)
from summarization_project.evaluation import compute_rouge
from summarization_project.generation import greedy_decode
from summarization_project.model import TransformerSummarizer
from summarization_project.preprocessing import clean_and_truncate, clean_text, truncate_context_window
from summarization_project.tokenizer_utils import train_bpe_tokenizer, token_id
from summarization_project.training import set_seed, train_model

config = ExperimentConfig()
set_seed(config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
train_examples, val_examples, test_examples = load_cnn_dailymail_splits(config)
print('Train / validation / test sizes:', len(train_examples), len(val_examples), len(test_examples))
print('Sample article:', train_examples[0].article[:700])
print('Sample summary:', train_examples[0].summary)

## Section 2: Data Cleaning
The cleaning pipeline removes HTML tags, malformed unicode, non-ASCII symbols, duplicate spaces, and normalizes quotation marks. Context-window truncation is necessary because Transformer self-attention scales quadratically with sequence length; keeping the first 256 and last 256 tokens preserves the lead and concluding context.

In [ ]:
dirty = '<p>“Breaking” news — caf\u00e9 &amp; markets   rally…</p>\n\nFinal line.'
print('Before:', dirty)
print('After :', clean_text(dirty))
long_text = ' '.join(f'tok{i}' for i in range(530))
truncated = truncate_context_window(long_text, max_tokens=512)
print('Truncated length:', len(truncated.split()))
print('Beginning:', truncated.split()[:5])
print('Ending:', truncated.split()[-5:])

## Section 3: Tokenizer Training
A custom BPE subword tokenizer is trained only on the training corpus with a vocabulary in the required 8,000-16,000 range. Word-level tokenization fails because rare political names, company names, and locations become OOV; BPE can compose names such as *Zelenskyy*, *Nvidia*, or *Ouagadougou* from reusable subword pieces.

In [ ]:
tokenizer = train_bpe_tokenizer(
    [ex.article for ex in train_examples] + [ex.summary for ex in train_examples],
    config.tokenizer_path,
    vocab_size=config.vocab_size,
    min_frequency=config.min_frequency,
)
specials = SpecialTokens()
pad_id = token_id(tokenizer, specials.pad)
bos_id = token_id(tokenizer, specials.bos)
eos_id = token_id(tokenizer, specials.eos)
print('Vocabulary size:', tokenizer.get_vocab_size())
for sample in ['President Zelenskyy met Nvidia executives in Ouagadougou.', 'OpenAI and Microsoft announced a project in Seattle.']:
    encoding = tokenizer.encode(sample)
    print(sample, '->', encoding.tokens)

In [ ]:
train_dataset = SummarizationDataset(train_examples, tokenizer, config)
val_dataset = SummarizationDataset(val_examples, tokenizer, config)
test_dataset = SummarizationDataset(test_examples, tokenizer, config)
train_loader = make_dataloader(train_dataset, pad_id, config.batch_size, shuffle=True)
val_loader = make_dataloader(val_dataset, pad_id, config.batch_size, shuffle=False)
batch = next(iter(train_loader))
print('Dynamic padded source shape:', batch['source_ids'].shape)
print('Actual source lengths:', batch['source_lengths'])

## Section 4: Model Architecture
The model uses token embeddings, sinusoidal positional encodings, explicit query/key/value projections, head splitting/concatenation, residual connections, feed-forward layers, and layer normalization. Cross-attention uses decoder hidden states as queries and encoder outputs as keys/values, producing `(batch, target_length, d_model)` decoder states.

In [ ]:
model = TransformerSummarizer(
    vocab_size=tokenizer.get_vocab_size(),
    pad_id=pad_id,
    d_model=config.d_model,
    num_heads=config.num_heads,
    d_ff=config.d_ff,
    num_layers=config.num_layers,
    dropout=config.dropout,
    max_length=max(config.max_article_tokens, config.max_summary_tokens) + 8,
).to(device)
logits = model(batch['source_ids'].to(device), batch['decoder_input_ids'].to(device))
print('Logits shape (batch, target_length, vocab):', tuple(logits.shape))

## Section 5: Masking
Padding masks prevent attention to PAD tokens and have shape `(batch_size, 1, 1, source_length)`. Causal masks are lower-triangular boolean matrices with shape `(target_length, target_length)` that prevent future-token visibility.

In [ ]:
import matplotlib.pyplot as plt

padding_mask = model.make_padding_mask(batch['source_ids'].to(device), pad_id)
causal_mask = model.make_causal_mask(12, device)
print('Padding mask shape:', tuple(padding_mask.shape))
print('Causal mask shape:', tuple(causal_mask.shape))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(padding_mask[0, 0].detach().cpu(), cmap='viridis', aspect='auto')
axes[0].set_title('Padding Mask')
axes[1].imshow(causal_mask.detach().cpu(), cmap='viridis')
axes[1].set_title('Causal Mask')
plt.tight_layout()

## Section 6: Training
Training uses teacher forcing: decoder input is `<BOS> + summary[:-1]` and the target is `summary[1:]`. Loss is cross entropy with label smoothing factor `0.1`, AdamW, gradient clipping, and a learning-rate scheduler. The default is the required minimum of 3 epochs.

In [ ]:
history = train_model(model, train_loader, val_loader, pad_id, config, device)
plt.figure(figsize=(5, 3))
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

## Section 7: Inference
Greedy decoding starts with `<BOS>`, chooses the highest-probability token at each step, appends it, and stops at `<EOS>` or 64 generated tokens.

In [ ]:
sample_summary = greedy_decode(
    model, tokenizer, test_examples[0].article, bos_id, eos_id, pad_id,
    config.max_article_tokens, config.max_generated_tokens, device,
)
print(sample_summary)

## Section 8: Evaluation
The next cell evaluates at least five unseen CNN/DailyMail test articles and reports article snippets, references, generated summaries, and ROUGE-1/2/L.

In [ ]:
predictions = []
references = []
for i, example in enumerate(test_examples[:5], start=1):
    generated = greedy_decode(model, tokenizer, example.article, bos_id, eos_id, pad_id, config.max_article_tokens, config.max_generated_tokens, device)
    predictions.append(generated)
    references.append(example.summary)
    print(f'ARTICLE {i} SNIPPET:', example.article[:500])
    print('REFERENCE:', example.summary)
    print('GENERATED:', generated)
    print('-' * 80)
rouge_scores = compute_rouge(predictions, references)
rouge_scores

## Section 9: Error Analysis
Because the default model is trained on a deliberately tiny CPU-friendly subset, generated summaries may show repetition, miss secondary facts, stop early, or hallucinate plausible news words. These issues are expected from limited data/model scale, but the implementation exposes the required mechanisms needed to improve quality by increasing the training subset, model dimensions, and epochs.